## Data Ingestion

In [ ]:
from langchain_core.documents import Document

In [ ]:
# doc = Document(    #document is a class, doc is a variable , doc is an instance of Document
#     page_content= "This is the main text content iam using to create RAG",
#     metadata= {
#         "source":"example.txt",
#         "pages":1,
#         "author":"Manish Tanvashi",
#         "Date created":"2026-02-17" 
#     }
# )
# doc  ## doc.__repr__() Actually like this 

In [ ]:
# sample_texts = {
#     "../data/text_files/python_intro.txt": """Python Programming Introduction

# Python is a high-level, interpreted programming language known for its simplicity and readability.
# Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
# programming languages in the world.

# Key Features:
# - Easy to learn and use
# - Extensive standard library
# - Cross-platform compatibility
# - Strong community support

# Python is widely used in web development, data science, artificial intelligence, and automation.
# """,

#     "../data/text_files/machine_learning.txt": """Machine Learning Basics

# Machine learning is a subset of artificial intelligence that enables systems to learn and improve
# from experience without being explicitly programmed. It focuses on developing computer programs
# that can access data and use it to learn for themselves.

# Types of Machine Learning:
# 1. Supervised Learning: Learning with labeled data
# 2. Unsupervised Learning: Finding patterns in unlabeled data
# 3. Reinforcement Learning: Learning through rewards and penalties
# """
# }

# for filepath,content in sample_texts.items():        ## .items() returns (key,value)
#     with open(filepath,"w",encoding= "utf-8") as f:
#         f.write(content)
# print("Sample Text CREATED.")

In [ ]:
# ## TextLoader

# from langchain_community.document_loaders import TextLoader

# loader = TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
# document = loader.load()
# print(document)

In [ ]:
### Directory Loader

from langchain_community.document_loaders import  DirectoryLoader, PyPDFLoader, PyMuPDFLoader

## Load all the text files from the directory 
pdf_loader = DirectoryLoader(
    "../Data/PDFs",
    ## Pattern to match files
    glob = "**/*.pdf",
    ## Loader class to use 
    loader_cls = PyMuPDFLoader,
    show_progress= False
)
documents = pdf_loader.load()
documents

## Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\nExample chunk:")
        print(split_docs[0].page_content[:200])
        print(split_docs[0].metadata)

    return split_docs

import langchain_text_splitters
all_pdf_documents = pdf_loader.load()

chunks = split_documents(all_pdf_documents)

chunks


In [ ]:
sources = set([chunk.metadata["source"] for chunk in chunks])

print("Total unique files:", len(sources))

for s in sources:
    print(s)

## Embedding and VectorStoreDB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """HANDELS DOCUMENT EMBEDDING GENERATION USING SENTENCE_TRANSFORMERS"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Initialize the Embedding_Manager
        Args:
             model_name : HuggingFace model name for sentence embeddings 
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the StentenceTransformer Model"""
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate Embeddings for a List of Texts
        
        Args:
             texts: List of text strings to embed
        
        returns: 
             numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model Not Loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")   
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generated embeddings with shape: {embeddings.shape} ")
        return embeddings
  
  ## Initialize Embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager

## Vector Store

In [ ]:
import os
class VectorStore:
    """Manages document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vectoe store
        
        Args:
             Collection_name: Name of the chromaDB collection
             persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        """

        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise
vectorstore = VectorStore()
vectorstore

In [ ]:
chunks

In [ ]:
## Convert texts to embeddings

texts = [doc.page_content for doc in chunks]

## Generate the embeddings 

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database

vectorstore.add_documents(chunks,embeddings)


## Retriever Pipeline from VectorStore 

In [ ]:
class RAGRetriever:
    """Handles query based retrieval from vector store """

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        """ Initialize the retriever
        Args: 
            vector_store:  Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in Vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            documents = results.get("documents", [[]])[0]
            metadatas = results.get("metadatas", [[]])[0]
            distances = results.get("distances", [[]])[0]
            ids = results.get("ids", [[]])[0]

            print("DEBUG → docs returned:", len(documents))

            for i,(doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                # Convert distance to similarity score (ChromaDB uses cosine distance)
                similarity_score = 1 - distance

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "distance": distance,
                    "rank": i + 1
                })
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")


            return retrieved_docs
    
        except Exception as e:
            print(f"Error during Retrieval: {e}")
            return[]

rag_retriever = RAGRetriever(vectorstore , embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("What is pineapple fermentation ?")

## Integration VectorDB context pipeline with LLM output

In [ ]:
## Simple RAG pipeline with GROQ LLM 
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the GROQ LLM (Set you API key in the environment)
groq_api_key = os.getenv("GROQ_API_KEY")

#Initilize llm model
llm = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.3-70b-versatile", temperature = 0.1, max_tokens = 1024)

# Simple RAG function: Retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=20):
    ## Retriever the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else""
    if not context:
        return "NO relevent context found to answer the question."
    
    ## Generate the answer using GROQ LLM 
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question:{query}

        Answer:"""
    
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [ ]:
answer = rag_simple("what is kimchi ?", rag_retriever, llm)
print(answer)

In [ ]:
print("Total documents loaded:", len(documents))

sources = set([doc.metadata.get("source") for doc in documents])
print("Files detected:")
for s in sources:
    print(s)

In [ ]:
docs = rag_retriever.retrieve("tepache", top_k=10)
for d in docs:
    print(d["content"][:300])
    print("-----")

In [ ]:
docs = rag_retriever.retrieve("tepache", top_k=5)

for d in docs:
    print(d["metadata"]["source"])